# HumanAI Detect - Uretici-Bazli SHAP Kiyaslamasi (Colab)

**Amac:** Yerel makinede tam veriyle (11896 ornek x 815 ozellik x 3 sinif) TreeExplainer 25+ dakikada bitmedi, durduruldu. Bu notebook (1) sinif-dengeli kucuk bir alt-ornek uretir (`make_shap_subsample.py`, varsayilan 700/sinif ~2100 toplam), (2) o alt-orneklem uzerinde SHAP hesaplayip HANGI OZELLIKLERIN HER URETICIDE (Qwen/GPT-4o-mini/Claude) tutarli onemli oldugunu, hangilerinin sadece tek ureticiye ozgu oldugunu tabloyla gosterir (`explain_by_generator.py`) -- proje notlarindaki 'kl_div_word_freq Qwen'de guclu, GPT-4o-mini'de neredeyse sifir' bulgusunun sistematik/tam versiyonu.

**Not:** Bu adim GPU'dan fayda gormez (SHAP TreeExplainer CPU-bound), yine de Colab'in guclu CPU/RAM'i yerel makineden daha hizli olacaktir. Runtime tipi onemli degil, varsayilan (hatta GPU'suz) yeterli.

**Girdi:** `data/processed/fused.parquet`, `outputs/models/xgboost_shap_current.pkl` (guncel 815-ozellik/11896-ornek veriyle egitilmis tani modeli -- yoksa `train_model.py --model xgboost --tag shap_current` ile once uretilmeli).
**Cikti:** `shap_by_generator.md/.csv`, `generator_comparison_top15.png`.

In [ ]:
# Repo klonla (ilk kez) veya guncelle (klasor zaten varsa git pull)
GITHUB_REPO = 'https://github.com/BurakCANKURT/humanai-detect.git'
import os
if os.path.exists('/content/humanai_detect'):
    %cd /content/humanai_detect
    !git pull origin master
else:
    !git clone {GITHUB_REPO} /content/humanai_detect
    %cd /content/humanai_detect
!git log --oneline -5


In [ ]:
# Bagimliliklari kur (birkac dakika surebilir)
!pip install -e '.[dev]' -q


In [ ]:
# fused.parquet + xgboost_shap_current.pkl yukle -- IKISINI BIRDEN sec
import os
os.makedirs('data/processed', exist_ok=True)
os.makedirs('outputs/models', exist_ok=True)
from google.colab import files
uploaded = files.upload()
import shutil
for fname in uploaded:
    if fname.endswith('.parquet'):
        shutil.move(fname, f'data/processed/{fname}')
    elif fname.endswith('.pkl'):
        shutil.move(fname, f'outputs/models/{fname}')
    else:
        print(f'beklenmeyen dosya, elle tasi: {fname}')


In [ ]:
# 1) Sinif-dengeli alt-ornek uret (label+generator korunur)
!python scripts/make_shap_subsample.py --n-per-class 700


In [ ]:
# 2) Uretici-bazli SHAP kiyaslamasi (asil is, birkac dakika surebilir)
!python scripts/explain_by_generator.py --model outputs/models/xgboost_shap_current.pkl --top-n 20


In [ ]:
# Sonucu ekranda goster
print(open('outputs/reports/shap_by_generator.md', encoding='utf-8').read())


In [ ]:
from IPython.display import Image, display
display(Image('outputs/figures/shap/generator_comparison_top15.png'))


In [ ]:
# Sonuclari yerel makineye indir
from google.colab import files
files.download('outputs/reports/shap_by_generator.md')
files.download('outputs/reports/shap_by_generator.csv')
files.download('outputs/figures/shap/generator_comparison_top15.png')
files.download('data/processed/shap_subsample.parquet')


## Indirilen Dosyalari Yerlestirme

`shap_by_generator.md/.csv` -> `outputs/reports/`, PNG -> `outputs/figures/shap/`, `shap_subsample.parquet` -> `data/processed/` (tekrar SHAP calistirmadan once alt-orneklemi yeniden uretmemek icin saklanabilir). Claude'a haber ver -- 'Tum ureticilerde top-N mi?' sutunu HAYIR olan ozellikler raporda 'uretici-ozgu, genellenemez sinyal' olarak, EVET olanlar 'genellenebilir aday' olarak tartisilabilir.